In [76]:
import qr
import importlib
importlib.reload(qr)

<module 'qr' from '/home/lexu/University/visible-light/inbedded/qr.py'>

In [77]:
import math
import numpy as np
import segno
import cv2
import moderngl

In [78]:
def create_qr(content):
    qr = segno.make_qr(content, version=10, mode="byte")
    matrix = qr.matrix
    arr = np.array(matrix, dtype=int)

    qr.save("output/qr.png", scale=10)

In [79]:
qcd = cv2.QRCodeDetector()

import zxingcpp

def read_qr(img):
    results = zxingcpp.read_barcodes(
        img,
        formats=zxingcpp.BarcodeFormat.QRCode
    )
    return [result.bytes for result in results]

In [80]:
msg = "hello world"

for word in msg.split(" "):
    create_qr(word)
    print(read_qr(cv2.imread("output/qr.png")))

[b'hello']
[b'world']


In [81]:

from protocol import *

subtitle = SendSubtitle(
    currentTime=1000,
    subtitleStartTime=1200,
    subtitleEndTime=3500,
    sub_string="Hello World!"
)

print(encode_message(subtitle).hex(" "))

create_qr(encode_message(subtitle))

bg_img = cv2.imread('wide_cat.jpg')
fg_img = cv2.imread('output/qr.png', cv2.IMREAD_UNCHANGED)

h, w, _ = bg_img.shape
h_fg, w_fg = fg_img.shape

bg_img = cv2.cvtColor(bg_img, cv2.COLOR_BGR2RGBA)
fg_img = cv2.cvtColor(fg_img, cv2.COLOR_BGRA2RGBA)

ctx = moderngl.create_standalone_context()

bg_tex = ctx.texture((w, h), 4, bg_img.tobytes())
fg_tex = ctx.texture((w_fg, h_fg), 4, fg_img.tobytes())
out_tex = ctx.texture((w, h), 4)

02 01 01 03 03 e8 01 03 04 b0 01 03 0d ac 12 0c 48 65 6c 6c 6f 20 57 6f 72 6c 64 21 b6 6b bc 8c 00


In [82]:
compute_shader = ctx.compute_shader(
    """

#version 430

layout(local_size_x = 16, local_size_y = 16) in;

layout(rgba8, binding = 0) uniform image2D bgImage;
layout(rgba8, binding = 1) uniform image2D fgImage;
layout(rgba8, binding = 2) uniform image2D outImage;

uniform bool u_phase;
uniform float u_strength; // Try 0.02 (2% brightness change)

void main() {
    ivec2 coords = ivec2(gl_GlobalInvocationID.xy);

    ivec2 size = imageSize(outImage);

    if (coords.x >= size.x || coords.y >= size.y) {
        return;
    }

    vec3 bg = imageLoad(bgImage, coords).rgb;
    vec3 finalColor = bg;

    ivec2 fgSize = imageSize(fgImage);

    if (coords.x < fgSize.x && coords.y < fgSize.y) {
        vec3 fg = imageLoad(fgImage, coords).rgb;

        // Convert QR pixel to +1 or -1
        float qr = fg.r < 0.5 ? 1.0 : -1.0;

        // Alternate the modulation between frames
        float phase = u_phase ? 1.0 : -1.0;

        // Slightly increase or decrease brightness
        float offset = qr * phase * u_strength;

        finalColor = clamp(bg + vec3(offset), 0.0, 1.0);
    }

    imageStore(outImage, coords, vec4(finalColor, 1.0));
}

"""
)

In [83]:
bg_tex.bind_to_image(0, read=True, write=False)
fg_tex.bind_to_image(1, read=True, write=False)
out_tex.bind_to_image(2, read=False, write=True)

group_x = math.ceil(w / 16)
group_y = math.ceil(h / 16)

for i in range(2):
    compute_shader['u_phase'].value = i % 2
    compute_shader.run(group_x, group_y, 1)

    result_data = out_tex.read()
    result_img = np.frombuffer(result_data, dtype=np.uint8).reshape((h, w, 4))

    result_img = cv2.cvtColor(result_img, cv2.COLOR_RGBA2BGR)
    cv2.imwrite(f"output/overlay_{i}.png", result_img)

    cv2.imshow('Compute Shader Result', result_img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [84]:
def detect_screen(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Reduce noise
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    # Detect boundaries
    edges = cv2.Canny(gray, 50, 150)

    
    # Join small gaps in the screen boundary
    kernel = np.ones((5, 5), np.uint8)
    edges = cv2.morphologyEx(
        edges, cv2.MORPH_CLOSE, kernel
    )
    
    show_img(edges)
    contours, _ = cv2.findContours(
        edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE
    )

    h, w = frame.shape[:2]
    candidates = []

    for contour in contours:
        area = cv2.contourArea(contour)
        print(area)

        if area < 0.1 * w * h:
            continue

        perimeter = cv2.arcLength(contour, True)
        polygon = cv2.approxPolyDP(
            contour, 0.02 * perimeter, True
        )

        if len(polygon) == 4 and cv2.isContourConvex(polygon):
            candidates.append((area, polygon))

    if not candidates:
        print("none found")
        return None

    # Return largest quadrilateral
    return max(candidates, key=lambda x: x[0])[1].reshape(4, 2)


In [85]:


def show_img(img):
    cv2.namedWindow("output", cv2.WINDOW_NORMAL)    # Create window with freedom of dimensions
    imS = cv2.resize(img, (1920, 1080))                # Resize image
    cv2.imshow("output", imS)                       # Show image
    cv2.waitKey(0)                                  # Display the image infinitely until any keypress


In [86]:
img0 = cv2.imread("phone_media/wide_cat_black.jpg")
img1 = cv2.imread("phone_media/wide_cat_white.jpg")

screen_img = cv2.imread("phone_media/screen1.jpg")

corners = detect_screen(screen_img)
result = screen_img.copy()


if corners is not None:
    cv2.polylines(
        result,
        [corners],
        True,
        (0, 255, 0),
        3
    )

    print(f"Detected {len(corners)} points")


show_img(result)

def align_frames(frame_a, frame_b):
    # Grayscale
    a = cv2.cvtColor(frame_a, cv2.COLOR_BGR2GRAY)
    b = cv2.cvtColor(frame_b, cv2.COLOR_BGR2GRAY)

    # Downsample for speed
    scale = 4
    a = cv2.resize(a, None, fx=1/scale, fy=1/scale)
    b = cv2.resize(b, None, fx=1/scale, fy=1/scale)

    # Phase correlation requires float32
    a = a.astype(np.float32)
    b = b.astype(np.float32)

    # Estimate translation from A to B
    (dx, dy), response = cv2.phaseCorrelate(a, b)

    # Apply inverse translation to B
    M = np.float32([
        [1, 0, -dx * scale],
        [0, 1, -dy * scale]
    ])

    h, w = frame_b.shape[:2]

    return cv2.warpAffine(frame_b, M, (w, h))

aligned_1 = align_frames(img0, img1)

dif = cv2.absdiff(img0, aligned_1)
dif = 255 - dif
cv2.imshow('Compute Shader Result', dif)
cv2.waitKey(0)
cv2.destroyAllWindows()

decoder = MessageDecoder()
for data in read_qr(dif):
    print(data.hex(" "))
    for message in decoder.push(data):
        print(message)

6.0
32.0
78.0
30.5
34.0
46.0
152.5
40.0
100.5
44.0
34.0
44.0
51.0
96.0
72.0
34.0
46.0
112.0
259.0
67.0
26.5
66.5
51.0
40.0
40.0
96.0
111.5
46.0
96.5
75.0
229.5
34.0
52.0
212.5
729.0
51.0
34.0
50.0
281.5
112.5
34.0
69.0
97.0
40.0
71.5
408.5
40.0
32.5
433.0
141.5
51.5
72.0
71.0
63.5
34.0
34.0
50.0
93.5
20.0
34.0
40.0
117.0
479.5
94.0
50.0
64.0
34.0
44.0
34.0
34.0
116.0
40.0
112.0
34.0
88.0
34.0
40.0
84.0
47.0
34.0
51.5
72.5
34.0
52.0
34.0
77.5
93.5
44.0
64.5
114.5
433.0
92.5
158.0
40.0
119.5
40.0
328.5
40.0
118.5
174.0
93.0
103.5
105.5
145.5
57.0
132.5
398.5
58.0
527.5
34.0
63.5
447.5
147.5
40.0
400.0
51.0
344.5
54.5
79.0
160.5
63.0
260.0
40.0
10.5
180.0
57.5
179.0
223.5
44.0
53.0
89.5
8.5
61.0
40.0
40.0
34.0
34.0
40.0
462.5
131.5
21.0
119.5
44.5
4.0
110.0
278.5
114.5
108.0
34.0
34.0
50.5
52.5
45.5
40.0
34.0
92.0
457.0
52.0
41.5
166.0
85.0
388.5
34.0
157.5
357.5
151.0
45.5
66.0
173.5
108.5
87.5
94.0
116.5
34.0
34.5
19.0
101.0
34.0
34.0
34.0
115.0
46.0
46.0
315.5
34.0
40.0
34.0
268.5
180.

In [ ]:
import cv2

camera_index = 4

cap = cv2.VideoCapture(camera_index)

decoder = MessageDecoder()

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)

fps = cap.get(cv2.CAP_PROP_FPS)
print(f"Camera FPS: {fps}")

if not cap.isOpened():
    print(f"Error: Could not open Iriun stream on index {camera_index}")
    exit()

print("Press 'q' to quit.")

while True:
    # Capture frame-by-frame
    ret, frame = cap.read()
    
    if not ret:
        print("Error: Failed to grab frame.")
        break
    
    for data in read_qr(frame):
        for message in decoder.push(data):
            print(message)

    # Display the resulting frame
    cv2.imshow('Iriun iPhone Stream', frame)

    # Break loop on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Clean up
cap.release()
cv2.destroyAllWindows()


Camera FPS: 30.0
Press 'q' to quit.
02 01 01 03 03 e8 01 03 04 b0 01 03 0d ac 12 0c 48 65 6c 6c 6f 20 57 6f 72 6c 64 21 b6 6b bc 8c 00
SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
02 01 01 03 03 e8 01 03 04 b0 01 03 0d ac 12 0c 48 65 6c 6c 6f 20 57 6f 72 6c 64 21 b6 6b bc 8c 00
SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
02 01 01 03 03 e8 01 03 04 b0 01 03 0d ac 12 0c 48 65 6c 6c 6f 20 57 6f 72 6c 64 21 b6 6b bc 8c 00
SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
02 01 01 03 03 e8 01 03 04 b0 01 03 0d ac 12 0c 48 65 6c 6c 6f 20 57 6f 72 6c 64 21 b6 6b bc 8c 00
SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
02 01 01 03 03 e8 01 03 04 b0 01 03 0d ac 12 0c 48 65 6c 6c 6f 20 57 6f 72 6c 64 21 b6 6b bc 8c 00
SendSubtitle(currentTime=1000, subtitleStartTime=1200